# Validation — `ternary-adapt-metamathqa-rank32`

**What this measures:** The claim names MetaMathQA and this repo publishes comparable rows from its own harness, so the target is the raw harness field `num_trainable_params` from a real `ternary_adapt` run at the llama-3.2-3B-rank32 protocol, with `test_accuracy` from the same run guarding the "without losing fit" half.

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`a4127624b333`](https://github.com/mayorquinmachines/peft/commit/a4127624b333eb6beda1a7498d62c02331b863ba)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/ternary_adapt/llama-3.2-3B-rank32` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

## 1. Environment

On Remyx compute this ran in `pytorch/pytorch:2.14.0-cuda12.6-cudnn9-runtime`; the exact Dockerfile is committed at `.remyx/validations/ternary-adapt-metamathqa-rank32.Dockerfile` — edit it and the next run builds from your version.

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout.

In [ ]:
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = "a4127624b333eb6beda1a7498d62c02331b863ba"

!git clone --quiet $REPO_URL repo
%cd repo
!git fetch --quiet --depth=1 origin $COMMIT && git checkout --quiet $COMMIT
!git log -1 --oneline
%pip install --quiet -e .

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/ternary_adapt/llama-3.2-3B-rank32` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/ternary_adapt/llama-3.2-3B-rank32/adapter_config.json`:

```json
{"auto_mapping": null, "base_model_name_or_path": "meta-llama/Llama-3.2-3B", "revision": null, "task_type": "CAUSAL_LM", "peft_type": "TERNARY_ADAPT", "inference_mode": false, "target_modules": ["q_proj", "v_proj"], "exclude_modules": null, "modules_to_save": null, "layers_to_transform": null, "layers_pattern": null, "block_shape": null, "ternarize_base": true, "fan_in_fan_out": false, "init_weights": true}
```

In [ ]:
print(open("method_comparison/MetaMathQA/experiments/ternary_adapt/llama-3.2-3B-rank32/adapter_config.json").read())

## 5. Confirm the change under test is what is loaded

The harness needs these modules imported before it reads the configuration; their paths must point into the checkout above.

In [ ]:
import importlib
m = importlib.import_module("peft.tuners.ternary_adapt"); print("peft.tuners.ternary_adapt", "→", m.__file__)
!git rev-parse HEAD

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/ternary_adapt/llama-3.2-3B-rank32` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
%cd /content/repo/method_comparison/MetaMathQA
import glob, importlib, runpy, sys
for m in ["peft.tuners.ternary_adapt"]:
    importlib.import_module(m)
configs = sorted(glob.glob("experiments/ternary_adapt/llama-3.2-3B-rank32/*/")) or ["experiments/ternary_adapt/llama-3.2-3B-rank32"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

## 7. Read what the benchmark wrote

Results land under `results/*.json` (relative to `method_comparison/MetaMathQA`). The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["results/*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 1146840,
        "baseline": 9174720
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.4,
        "baseline": null
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    print(f"{c['metric']:<28}{str(v):>16}{str(c['baseline']):>16}  {c['direction']} {t}  {mark}")

## 9. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 1146840 with `test_accuracy` >= 0.4 holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
# Suite kind (a): the repo's own MetaMathQA harness. The claim names MetaMathQA and this repo publishes comparable
# numbers from exactly this runner, so a synthesized CPU surrogate would answer a different question. The published
# LoRA r=32 corpus row IS the baseline; no baseline arm runs. Metric keys are copied character-for-character from
# the harness's result schema (num_trainable_params, test_accuracy).
benchmarks:
  - name: ternary-adapt-metamathqa-rank32
    suite:
      harness:
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/ternary_adapt/llama-3.2-3B-rank32  # NEW config dir for the NEW method (see EXPERIMENT_FILES)
        results_glob: "method_comparison/MetaMathQA/results/*.json"
        method: ternary_adapt
        # The PR confines registration to the new package's __init__ (register_peft_method + a dynamically added
        # PeftType member execute on import; peft/tuners/__init__.py does not import it), so the runner must
        # import peft.tuners.ternary_adapt in-process before loading adapter_config.json with peft_type TERNARY_ADAPT.
        preimport:
          - "peft.tuners.ternary_adapt"
      scorer: num_trainable_params
    metrics:
      # TARGET: "far fewer trainable parameters than standard LoRA" — raw harness field, lower is better.
      # Derivation: published LoRA r=32 q/v row = 28 x [32*(3072+3072) + 32*(3072+1024)] = 9,174,720 trainable.
      # TernaryAdapt default near-square Kronecker blocks = 28 x [(64*64+48*48) + (32*64+32*48)] = 279,552
      # (~32.8x fewer). Threshold 1,146,840 = 9,174,720 / 8: demands at least 8x fewer params than the published
      # LoRA row while sitting ~4.1x above the derived ternary value, so a correct implementation clears it.
      - name: num_trainable_params
        direction: min
        threshold: 1146840
        role: target
      # GUARDRAIL: "without losing fit on MetaMathQA" — the fit half of the claim, from the SAME harness run.
      # Floor 0.40 (test_accuracy is a 0-1 fraction in this harness's results JSONs): the published LoRA r=32 row
      # clears it comfortably (a successful MetaMathQA SFT of a 3B model), while a broken ternarization
      # (collapsed base weights or a dead mask) falls toward untuned-base 0-shot levels and fails, so this is a
      # real no-regression bound the baseline already satisfies.
      - name: test_accuracy
        direction: max
        threshold: 0.4
        role: guardrail
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        num_trainable_params: 9174720
    policy:
      guardrail_veto: true
    compute:
      # ONE arm = llama-3.2-3B SFT at the harness's default MetaMathQA protocol (~2-5k optimizer steps x ~1.5 s/step
      # on one A100-class GPU) + generation-based test_accuracy eval + ~7 GB base-model download ≈ 2-4 h;
      # 21600 s adds honest download/scheduling headroom.
      tier: gpu
      timeout_s: 21600
    held_constant:
      - "base model meta-llama/Llama-3.2-3B at the llama-3.2-3B-rank32 protocol (same as every published corpus row)"
      - "training protocol: the harness's default training params (no per-experiment training_params.json override)"
      - "target modules q_proj+v_proj, identical to the comparable LoRA r=32 corpus row"
      - "MetaMathQA train/eval data and seed fixed by run.py, same for every published row"
    avoid:
      - "unpinned base-model revision"
      - "a per-experiment training_params.json that overrides the published default protocol and breaks comparability"
      - "pointing experiments at another method's directory (would measure that method, not ternary_adapt)"
    provenance:
      num_trainable_params: "user_guidance"
      test_accuracy: "user_guidance"
      baseline: "published_results:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json (LoRA r=32 q/v row = 9,174,720 trainable, per the PR derivation)"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      held_constant: "protocol_doc:method_comparison/README.md"
      preimport: "upstream_comment:PR diff — registration runs only on import of peft.tuners.ternary_adapt"
      experiments: "synthesized: new config dir for the new method, mirroring experiments/adalora/llama-3.2-3B-rank32 shape"
```